In [1]:
import pandas, urllib, numpy

In [2]:
import statsmodels, statsmodels.stats, statsmodels.stats.multitest

In [3]:
import matplotlib, matplotlib.pyplot
matplotlib.rcParams.update({'font.size':20, 
                            'font.family':'sans-serif', 
                            'xtick.labelsize':16, 
                            'ytick.labelsize':16, 
                            'figure.figsize':(16*(2/3), 9*(2/3)), 
                            'axes.labelsize':20
                           })
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [4]:
# pip install git+https://github.com/Maayanlab/maayanlab-bioinformatics.git
import maayanlab_bioinformatics, maayanlab_bioinformatics.enrichment

# define library

In [5]:
enrichr_library = 'Reactome_Pathways_2024'

# read data

In [10]:
df = pandas.read_csv('029 orsay Table DEGs - both HOM & VGA.tsv', sep='\t')
original = df['Gene name'].values
gene_list_input = []
for element in original:
    print(element)
    gene_list_input.append(element.upper())
print(len(gene_list_input), gene_list_input[:10])

Plbd1
Hp
Ly6c2
Sbsn
Stxbp2
Aqp3
Col7a1
Lsr
F2rl1
H2-Ab1
Urah
Klk7
9630013D21Rik
Asprv1
Krt17
Ass1
Lad1
Krt6a
Ly6a
Ptprc
Lgals7
Sfn
Ctss
Cidec
Fcgr4
Fgfr3
Wif1
Rab25
Anxa8
C3
Gm28967
Pkp3
Col6a5
Trbc2
Psapl1
Sprr1a
Serinc2
Apoc1
Adipoq
Klf4
Shtn1
Ccr2
Klc3
Dmkn
Krt16
S100a14
Pkp1
Lgals3
Siglec1
nan


AttributeError: 'float' object has no attribute 'upper'

In [9]:
for element in original:
    print(element)

Plbd1
Hp
Ly6c2
Sbsn
Stxbp2
Aqp3
Col7a1
Lsr
F2rl1
H2-Ab1
Urah
Klk7
9630013D21Rik
Asprv1
Krt17
Ass1
Lad1
Krt6a
Ly6a
Ptprc
Lgals7
Sfn
Ctss
Cidec
Fcgr4
Fgfr3
Wif1
Rab25
Anxa8
C3
Gm28967
Pkp3
Col6a5
Trbc2
Psapl1
Sprr1a
Serinc2
Apoc1
Adipoq
Klf4
Shtn1
Ccr2
Klc3
Dmkn
Krt16
S100a14
Pkp1
Lgals3
Siglec1
nan
Zfp991
Dgat2
Perp
Lrrc15
Oasl1
Dpyd
Cdh1
Cxcl10
Tacstd2
Vdr
Ifi209
Igfbp2
Ly6d
Cdo1
Irf7
Ppp1r13l
Krt5
Wfdc21
Trim29
Dsp
Emb
Vax2
Krt80
Cybb
Krt14
Tns4
Myo1g
Epcam
Endou
Zic1
Trp63
Ckmt1
Ly6g6c
Nectin4
Tcfl5
Sorl1


In [ ]:
def get_library(lib_name):
    '''
    Returns a dictionary mapping each term from the input library to 
    its associated geneset. 
    '''
    raw_lib_data = []

    with urllib.request.urlopen('https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=' + lib_name) as f:
        for line in f.readlines():
            raw_lib_data.append(line.decode("utf-8").split("\t\t"))

    name = []
    gene_list = []
    lib_data = {}

    for i in range(len(raw_lib_data)):
        name += [raw_lib_data[i][0]]
        raw_genes = raw_lib_data[i][1].replace('\t', ' ')
        gene_list += [raw_genes[:-1]]
    
    lib_data = {a[0]:a[1].split(' ') for a in zip(name, gene_list)}
    return lib_data

lib_data = get_library(enrichr_library)

enrich_results = maayanlab_bioinformatics.enrichment.enrich_crisp(gene_list_input, lib_data, 21000, True)

res_df = pandas.DataFrame(
    [ [
        term, 
        res.pvalue, 
        res.odds_ratio,
        res.n_overlap,

    ] for (term, res) in enrich_results ], 
    columns=['term', 'pvalue', 'odds_ratio', 'n_overlap']
)
res_df

In [ ]:
# add gene names
container = []
for pathway in res_df['term']:
    
    overlap_genes = []
    for query in lib_data[pathway]:
        if query in gene_list_input:
            overlap_genes.append(query)
    mouse_overlap_genes = [element.lower().title() for element in overlap_genes]
    overlap_genes_string = ', '.join(mouse_overlap_genes)
    container.append(overlap_genes_string)

res_df.loc[:, 'overlap_genes'] = container
res_df

In [ ]:
# multiple test correction
results = statsmodels.stats.multitest.multipletests(res_df['pvalue'], alpha=0.05, method='fdr_bh')
res_df['adjusted'] = results[1]

In [ ]:
# plot

def importance_calculator(x, y, w):

    xnorm = (x - numpy.min(x)) / (numpy.max(x) - numpy.min(x))
    ynorm = (y - numpy.min(y)) / (numpy.max(y) - numpy.min(y))
    wnorm = (w - numpy.min(w)) / (numpy.max(w) - numpy.min(w))
    
    importance = [numpy.sqrt(xnorm[i]**2 + ynorm[i]**2 + wnorm[i]**2) for i in range(len(x))]
    return importance

inflation = 25

a = res_df[(res_df['adjusted'] > 0.05) | (res_df['odds_ratio'] < 2)]
print(a.shape)
x = a['odds_ratio']
y = -numpy.log10(a['adjusted'])
s = a['n_overlap']
matplotlib.pyplot.scatter(x, y, s*inflation, alpha=1/10, edgecolors='none', color='black')

b = res_df[(res_df['adjusted'] < 0.05) & (res_df['odds_ratio'] > 2)]
print(b.shape)
x = b['odds_ratio']
y = -numpy.log10(b['adjusted'])
s = b['n_overlap']
sc = matplotlib.pyplot.scatter(x, y, s*inflation, alpha=1, edgecolors='none', color='pink')

x = list(b['odds_ratio'])
y = list(-numpy.log10(b['adjusted']))
prez = list(b['term'].values)
z = [element.split()[-1] for element in prez]
importance = importance_calculator(x, y, s.values)
b.loc[:, 'importance'] = importance
c = b.sort_values('importance', ascending=False)

threshold = importance[numpy.argsort(importance)[::-1][10]] # gimme the top 10
com = c[c['importance'] > threshold]
count = 0
for index, row in com.iterrows():
    x = row['odds_ratio']
    y = -numpy.log10(row['adjusted'])
    
    label = ' '.join(row['term'].split()[:2])
    label = row['term']
    matplotlib.pyplot.text(x, y, label, fontsize=8)
    count = count + 1
    print('{}\t{}\t{}\t{}'.format(count, row['importance'], row['term'], row['adjusted']))

matplotlib.pyplot.legend(*sc.legend_elements("sizes", num=3))
matplotlib.pyplot.xlabel('Odds ratio')
matplotlib.pyplot.ylabel('Significance [-log10 P]')
matplotlib.pyplot.grid(ls=':', alpha=1)
matplotlib.pyplot.tight_layout()

matplotlib.pyplot.show()
#matplotlib.pyplot.savefig('hom.svg')

In [ ]:
# store list 
print(c.shape)
c.drop('pvalue', axis='columns', inplace=True)
d = c.iloc[:, [0, 5, 4, 1, 2, 3]]
d.rename(columns={"term": "Term", 
                  "importance": "Priority", 
                  'adjusted':'Adjusted P', 
                  'odds_ratio':'Odds ratio',
                 'n_overlap':'N overlap',
                 'overlap_genes':'Overlap genes'}, inplace=True)
d.set_index('Term', inplace=True)
d.head()
d.to_csv('particularboth.wikipathways.tsv', sep='\t')

d